In [ ]:
!pip uninstall torchvision -y
!pip install whisperx
!pip install pyngrok
!pip install speechbrain

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import os
import gc
import re
import tempfile
from typing import Optional
from collections import deque

import numpy as np
import torch
import uvicorn
import whisperx
from fastapi import FastAPI, UploadFile, File, Header, HTTPException, Form
from whisperx.diarize import DiarizationPipeline
from speechbrain.inference.speaker import SpeakerRecognition
from google.colab import userdata
from pyngrok import ngrok

app = FastAPI()

# --- Настройки ---
API_KEY = userdata.get("API_KEY")
HF_TOKEN = userdata.get("HF_TOKEN")
DEVICE = "cuda"
MAX_FILE_SIZE_MB = 50

# Пороги для ECAPA-TDNN после исправления нормализации и склейки отрезков.
# Ожидаемые значения сходства одного человека: 0.75–0.95.
# Калибровка — строгий порог, не сливаем разных людей пока мало данных.
# После калибровки — мягче, реестр уже заполнен.
SIMILARITY_THRESHOLD_CALIBRATION = 0.55
SIMILARITY_THRESHOLD_MATCHED     = 0.42
SIMILARITY_UPDATE_MIN            = 0.50

# Минимальная суммарная длина голоса спикера для извлечения эмбеддинга.
# Меньше — эмбеддинг ненадёжен.
MIN_SPEAKER_DURATION = 1.5  # секунд

EMBEDDING_BUFFER_SIZE = 10


# ---------------------------------------------------------------------------
# Профиль спикера
# ---------------------------------------------------------------------------
class SpeakerProfile:
    def __init__(self, speaker_id: str, initial_embedding: np.ndarray):
        self.id = speaker_id
        self.centroid = initial_embedding.copy()
        self.buffer: deque = deque([initial_embedding.copy()], maxlen=EMBEDDING_BUFFER_SIZE)
        self.segment_count = 0

    @property
    def display_name(self) -> str:
        return self.id

    def vote_similarity(self, embedding: np.ndarray) -> float:
        """Среднее сходство по буферу — устойчивее чем один центроид."""
        if not self.buffer:
            return -1.0
        sims = [
            float(np.dot(e, embedding) / (np.linalg.norm(e) * np.linalg.norm(embedding) + 1e-8))
            for e in self.buffer
        ]
        return float(np.mean(sims)) if sims else -1.0

    def update(self, embedding: np.ndarray, sim: float):
        """Обновляем центроид только при уверенном совпадении."""
        self.buffer.append(embedding.copy())
        self.segment_count += 1
        if sim >= SIMILARITY_UPDATE_MIN:
            alpha = 0.05 + 0.10 * (sim - SIMILARITY_UPDATE_MIN)
            self.centroid = (1 - alpha) * self.centroid + alpha * embedding


# ---------------------------------------------------------------------------
# Реестр спикеров
# ---------------------------------------------------------------------------
profiles: dict[str, SpeakerProfile] = {}
speaker_counter = 0
session_max_speakers: Optional[int] = None


# ---------------------------------------------------------------------------
# Загрузка моделей
# ---------------------------------------------------------------------------
try:
    model = whisperx.load_model(
        "large-v2",
        DEVICE,
        compute_type="float16",
        vad_options={"vad_onset": 0.5, "vad_offset": 0.363}
    )
    diarize_model = DiarizationPipeline(token=HF_TOKEN, device=DEVICE)

    embedding_model = SpeakerRecognition.from_hparams(
        source="speechbrain/spkrec-ecapa-voxceleb",
        savedir="/tmp/ecapa",
        run_opts={"device": DEVICE}
    )
    print(">>> Модели успешно загружены.")
except Exception as e:
    raise RuntimeError(f"Ошибка загрузки моделей: {e}")


# ---------------------------------------------------------------------------
# Вспомогательные функции
# ---------------------------------------------------------------------------
def check_api_key(x_api_key: str):
    if x_api_key != API_KEY:
        raise HTTPException(status_code=403, detail="Invalid API Key")


def peak_normalize(chunk: np.ndarray) -> np.ndarray:
    """
    Нормализация по пику — приводит максимальную амплитуду к 1.
    В отличие от RMS-нормализации не выходит за диапазон [-1, 1],
    который ожидает ECAPA-TDNN.
    """
    peak = np.max(np.abs(chunk))
    if peak < 1e-8:
        return chunk
    return chunk / peak


def get_speaker_embedding(
    audio: np.ndarray,
    sample_rate: int,
    time_ranges: list[tuple[float, float]]
) -> Optional[np.ndarray]:
    """
    Извлекает эмбеддинг голоса спикера.

    Ключевое улучшение: все отрезки одного спикера склеиваются в один кусок
    перед подачей в ECAPA-TDNN. Это даёт модели достаточно голосового материала
    и стабилизирует эмбеддинг (короткие отрезки < 1-2 сек ненадёжны).
    """
    chunks = []
    total_duration = 0.0

    for start, end in time_ranges:
        duration = end - start
        if duration < 0.3:  # совсем короткий — пропускаем
            continue
        start_sample = int(start * sample_rate)
        end_sample = int(end * sample_rate)
        chunk = audio[start_sample:end_sample]
        chunks.append(chunk)
        total_duration += duration

    if total_duration < MIN_SPEAKER_DURATION or not chunks:
        return None

    # Склеиваем все отрезки в один — ECAPA получает максимум голосового материала
    combined = np.concatenate(chunks)

    # Peak-нормализация — не ломает диапазон [-1, 1] который ожидает ECAPA
    combined = peak_normalize(combined)

    try:
        waveform = torch.tensor(combined).unsqueeze(0).float()  # [1, samples]
        emb = embedding_model.encode_batch(waveform)
        return emb.squeeze().cpu().numpy()
    except Exception as e:
        print(f">>> Ошибка эмбеддинга: {e}")
        return None


def match_or_register(embedding: np.ndarray) -> SpeakerProfile:
    """Сопоставляет эмбеддинг с реестром или регистрирует нового спикера."""
    global speaker_counter

    best_profile = None
    best_sim = -1.0

    for profile in profiles.values():
        sim = profile.vote_similarity(embedding)
        if sim > best_sim:
            best_sim = sim
            best_profile = profile

    registry_full = (
        session_max_speakers is not None
        and len(profiles) >= session_max_speakers
    )
    threshold = SIMILARITY_THRESHOLD_MATCHED if registry_full else SIMILARITY_THRESHOLD_CALIBRATION

    if best_sim >= threshold and best_profile is not None:
        best_profile.update(embedding, best_sim)
        return best_profile
    else:
        if registry_full and best_profile is not None:
            print(f">>> [{best_sim:.2f}] реестр полон → {best_profile.display_name}")
            best_profile.update(embedding, best_sim)
            return best_profile
        # Новый спикер
        speaker_counter += 1
        new_id = f"Игрок_{speaker_counter}"
        profile = SpeakerProfile(new_id, embedding)
        profiles[new_id] = profile
        phase = "калибровка" if not registry_full else "сверх лимита"
        print(f">>> Новый спикер: {new_id} ({phase}, сходство: {best_sim:.2f})")
        return profile


def remap_speakers(segments: list, audio: np.ndarray, sample_rate: int = 16000) -> list:
    """Заменяет временные SPEAKER_XX на стабильные ID."""
    # Собираем все временные отрезки по временному ID
    speaker_segments: dict[str, list[tuple[float, float]]] = {}
    for seg in segments:
        spk = seg.get("speaker", "UNKNOWN")
        if spk == "UNKNOWN":
            continue
        speaker_segments.setdefault(spk, []).append((seg["start"], seg["end"]))

    # Для каждого временного ID — один эмбеддинг из склеенных отрезков
    temp_to_profile: dict[str, SpeakerProfile] = {}
    for temp_id, time_ranges in speaker_segments.items():
        emb = get_speaker_embedding(audio, sample_rate, time_ranges)
        if emb is None:
            print(f">>> {temp_id}: недостаточно голоса для эмбеддинга, пропускаем")
            continue
        profile = match_or_register(emb)
        temp_to_profile[temp_id] = profile
        print(f">>> {temp_id} → {profile.display_name}")

    # Заменяем ID в сегментах
    for seg in segments:
        spk = seg.get("speaker", "UNKNOWN")
        if spk in temp_to_profile:
            seg["speaker"] = temp_to_profile[spk].display_name

    return segments


# ---------------------------------------------------------------------------
# Эндпоинты
# ---------------------------------------------------------------------------
@app.post("/reset")
async def reset(x_api_key: str = Header(...)):
    check_api_key(x_api_key)
    global profiles, speaker_counter, session_max_speakers
    profiles = {}
    speaker_counter = 0
    session_max_speakers = None
    gc.collect()
    torch.cuda.empty_cache()
    print(">>> Память и реестр спикеров очищены. Новая игра.")
    return {"status": "memory cleared"}


@app.post("/process_chunk")
async def process_chunk(
    file: UploadFile = File(...),
    num_speakers: int = Form(None),
    min_speakers: int = Form(None),
    max_speakers: int = Form(None),
    x_api_key: str = Header(...)
):
    check_api_key(x_api_key)

    contents = await file.read()
    if len(contents) > MAX_FILE_SIZE_MB * 1024 * 1024:
        raise HTTPException(status_code=413, detail="File too large")

    with tempfile.NamedTemporaryFile(dir="/dev/shm", suffix=".wav", delete=False) as tmp:
        tmp.write(contents)
        temp_path = tmp.name

    try:
        audio = whisperx.load_audio(temp_path)

        if len(audio) < 32000:
            print(f">>> Чанк слишком короткий ({len(audio)} сэмплов), пропускаем.")
            return {"segments": []}

        # Транскрипция
        result = model.transcribe(audio, batch_size=16, language="ru")

        # Выравнивание
        model_a, metadata = whisperx.load_align_model(
            language_code=result["language"],
            device=DEVICE
        )
        result = whisperx.align(
            result["segments"], model_a, metadata, audio, DEVICE
        )
        del model_a
        gc.collect()
        torch.cuda.empty_cache()

        # Диаризация — диаризатор сам решает сколько голосов в чанке.
        # num_speakers не передаём — в каждом чанке реально говорят 1-3 человека,
        # не все N игроков. Глобальное разделение на N делает реестр эмбеддингов.
        diarize_kwargs = {}
        if min_speakers is not None:
            diarize_kwargs["min_speakers"] = min_speakers
        if max_speakers is not None:
            diarize_kwargs["max_speakers"] = max_speakers

        # Запоминаем лимит для реестра эмбеддингов (из num_speakers или max_speakers)
        global session_max_speakers
        if session_max_speakers is None:
            if num_speakers is not None:
                session_max_speakers = num_speakers
            elif max_speakers is not None:
                session_max_speakers = max_speakers
            if session_max_speakers is not None:
                print(f">>> Лимит реестра спикеров: {session_max_speakers}")

        try:
            diarize_segments = diarize_model(audio, **diarize_kwargs)
        except Exception as e:
            if "n_samples" in str(e) and "n_clusters" in str(e):
                print(">>> Мало сегментов, снимаем ограничение min_speakers")
                try:
                    fallback_kwargs = {}
                    if "max_speakers" in diarize_kwargs:
                        fallback_kwargs["max_speakers"] = diarize_kwargs["max_speakers"]
                    diarize_segments = diarize_model(audio, **fallback_kwargs)
                except Exception as e2:
                    print(f">>> Диаризация не удалась совсем: {e2}")
                    return {"segments": result["segments"]}
            else:
                print(f">>> Ошибка диаризации: {e}")
                return {"segments": result["segments"]}

        result = whisperx.assign_word_speakers(diarize_segments, result)

        # Сопоставление спикеров
        result["segments"] = remap_speakers(result["segments"], audio)

    except Exception as e:
        print(f">>> Ошибка обработки чанка: {e}")
        torch.cuda.empty_cache()
        return {"segments": []}

    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)
        torch.cuda.empty_cache()

    return {"segments": result["segments"]}


# --- Запуск ---
ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))
public_url = ngrok.connect(8000)
print(f">>> Публичный URL: {public_url}")

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()